# 25 - Disambiguating the conv win: scale vs. subword

At 30M params on WikiText-103 BPE, a depthwise conv beat the combo by ~10% (robust over 2 seeds).
At 3M params on TinyShakespeare CHAR it did not (+0.8%). Three things differed at once: model size,
tokenization (char->BPE), and block length (128->256). This notebook holds tokenization+block fixed
to the 30M setting (WT103 BPE, block 256) but shrinks the MODEL back to 3M, to test which factor
drives the conv win.

* If `combo_conv3 < combo` at **3M/BPE** too -> the win is about **subword data**, not model scale
  (a clean, scale-independent claim).
* If conv does **not** help at 3M/BPE -> the win is about **model scale**.

3M params (n_embd256, L4, H4), WT103 BPE, block 256, 3000 iters, 2 seeds. Per-config CSV resume.


In [1]:
%matplotlib inline
import os, math, time, csv, json
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 32
def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(over, seed=0, max_iters=3000, eval_iters=200, lr=3e-4, warmup=150, min_lr=3e-5):
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=4, n_head=4, n_embd=256,
                 dropout=0.1, **over)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    for it in range(max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return math.exp(ev()), count_params(model) / 1e6

print("ready")


device: cuda
vocab=8192  train=14,923,628  val=281,169
ready


In [2]:
COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
CONFIGS = {
    "combo":       COMBO,
    "combo_conv3": {**COMBO, "conv_kernel": 3},
    "combo_diff":  {**COMBO, "attention": "diff"},
}
SEEDS = [0, 1]

runs_csv = "../results/conv_disambig_runs.csv"
done = set()
if os.path.exists(runs_csv):
    with open(runs_csv) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 4: done.add((row[0], row[1]))
else:
    with open(runs_csv, "w", newline="") as f:
        csv.writer(f).writerow(["config", "seed", "params_M", "val_ppl"])

t0 = time.time()
for name, over in CONFIGS.items():
    for s in SEEDS:
        if (name, str(s)) in done: continue
        p, pm = train_one(over, seed=s)
        with open(runs_csv, "a", newline="") as f:
            csv.writer(f).writerow([name, s, f"{pm:.3f}", f"{p:.4f}"])
        print(f"  {name:12s} seed {s}: ppl {p:.3f} ({pm:.2f}M)  | {(time.time()-t0)/60:.1f} min", flush=True)
print(f"ALL DONE in {(time.time()-t0)/60:.1f} min")


  combo        seed 0: ppl 99.895 (5.25M)  | 2.4 min


  combo        seed 1: ppl 99.743 (5.25M)  | 4.8 min


  combo_conv3  seed 0: ppl 117.727 (5.26M)  | 7.4 min


  combo_conv3  seed 1: ppl 120.799 (5.26M)  | 10.0 min


  combo_diff   seed 0: ppl 123.173 (5.25M)  | 13.3 min


  combo_diff   seed 1: ppl 124.358 (5.25M)  | 16.6 min


ALL DONE in 16.6 min


In [3]:
import numpy as np, csv
rows = {}
with open("../results/conv_disambig_runs.csv") as f:
    r = csv.reader(f); next(r, None)
    for c, s, pm, pp in r:
        rows.setdefault(c, []).append(float(pp))
m = {c: float(np.mean(v)) for c, v in rows.items()}
print(f"3M params, WT103 BPE, block 256 (2 seeds):")
for c in ["combo", "combo_conv3", "combo_diff"]:
    print(f"  {c:12s} {m[c]:.3f}  ({100*(m[c]-m['combo'])/m['combo']:+.2f}% vs combo)")
conv_d = 100 * (m["combo_conv3"] - m["combo"]) / m["combo"]
print(f"\nCONV at 3M/BPE: {conv_d:+.2f}% vs combo")
print("=> conv win is about SUBWORD DATA (helps at 3M too), NOT model scale" if conv_d < -2
      else "=> conv win is about MODEL SCALE (does not help at 3M/BPE)" if conv_d > -1
      else "=> ambiguous / partial")
print("(compare: 30M/BPE conv was -9.8%/-10.6%; 3M/char conv was +0.8%)")


3M params, WT103 BPE, block 256 (2 seeds):
  combo        99.819  (+0.00% vs combo)
  combo_conv3  119.263  (+19.48% vs combo)
  combo_diff   123.766  (+23.99% vs combo)

CONV at 3M/BPE: +19.48% vs combo
=> conv win is about MODEL SCALE (does not help at 3M/BPE)
(compare: 30M/BPE conv was -9.8%/-10.6%; 3M/char conv was +0.8%)


## Reading this

This pins down the conv finding for the paper. Holding tokenization (BPE) and block (256) fixed and
only shrinking the model to 3M: if conv still clearly helps, the driver is **subword tokenization**,
giving a scale-independent claim ("a depthwise causal conv improves modern subword decoders"). If it
vanishes, the driver is **model scale**. Combined with 30M/BPE (-10%) and 3M/char (+0.8%), this gives
a clean, attributable story.
